In [1]:
import numpy as np

#### $p(x,a,y,z)=p(x|y)p(a|y)p(y|z)p(z)$ **(Model)** 

In [2]:
p_x_given_y = np.zeros((2, 2))
p_x_given_y[0, 0] = 0.5
p_x_given_y[0, 1] = 0.3
p_x_given_y[1, 0] = 0.5
p_x_given_y[1, 1] = 0.7

p_y_given_z = np.zeros((2, 2))
p_y_given_z[0, 0] = 0.1
p_y_given_z[0, 1] = 0.2
p_y_given_z[1, 0] = 0.9
p_y_given_z[1, 1] = 0.8

p_a_given_y = np.zeros((2, 2))
p_a_given_y[0, 0] = 0.3
p_a_given_y[0, 1] = 0.4
p_a_given_y[1, 0] = 0.7
p_a_given_y[1, 1] = 0.6

p_z = np.zeros(2)
p_z[0] = 0.4
p_z[1] = 0.6

#### Analytical

In [10]:
p_x_and_a_and_y_and_z = np.einsum('xy,ay,yz,z->xayz', p_x_given_y, p_a_given_y, p_y_given_z, p_z)

p_x = np.einsum('xayz->x', p_x_and_a_and_y_and_z)
p_a = np.einsum('xayz->a', p_x_and_a_and_y_and_z)
p_y = np.einsum('xayz->y', p_x_and_a_and_y_and_z)
p_z = np.einsum('xayz->z', p_x_and_a_and_y_and_z)

print('p(x) = ' + str(p_x))
print('p(a) = ' + str(p_a))
print('p(y) = ' + str(p_y))
print('p(z) = ' + str(p_z))

p(x) = [0.332 0.668]
p(a) = [0.384 0.616]
p(y) = [0.16 0.84]
p(z) = [0.4 0.6]


#### $q(x,a,y,y',y'',z)=\frac{q(x,y)q(a,y'')q(y,y',y'')q(y',z)}{q(y)q(y'')q(y')}, q(y,y',y'')=\delta(y-y')\delta(y-y'')$ **(Sum Product)**

In [16]:
l_mu_z = p_z
l_mu_yy = np.einsum('yz,z->y', p_y_given_z, l_mu_z)
u_mu_yyy = np.einsum('ay->y', p_a_given_y)
l_mu_y = np.einsum('y,y->y', l_mu_yy, u_mu_yyy)
l_mu_x = np.einsum('xy,y->x', p_x_given_y, l_mu_y)
r_mu_y = np.einsum('xy->y', p_x_given_y)
r_mu_yy = np.einsum('y,y->y', r_mu_y, u_mu_yyy)
r_mu_z = np.einsum('yz,y->z', p_y_given_z, r_mu_yy)
d_mu_yyy = np.einsum('y,y->y', r_mu_y, l_mu_yy)
d_mu_a = np.einsum('ay,y->a', p_a_given_y, d_mu_yyy)

q_x = (l_mu_x) / np.sum(l_mu_x)
q_a = (d_mu_a) / np.sum(d_mu_a)
q_y = (r_mu_y * l_mu_y) / np.sum(r_mu_y * l_mu_y)
q_yy = (r_mu_yy * l_mu_yy) / np.sum(r_mu_yy * l_mu_yy)
q_yyy = (d_mu_yyy * u_mu_yyy) / np.sum(d_mu_yyy * u_mu_yyy)
q_z = (r_mu_z * l_mu_z) / np.sum(r_mu_z * l_mu_z)

print('q(x) = ' + str(q_x))
print('q(a) = ' + str(q_a))
print('q(y) = ' + str(q_y))
print("q(y') = " + str(q_yy))
print('q(y") = ' + str(q_yyy))
print('q(z) = ' + str(q_z))

q(x) = [0.332 0.668]
q(a) = [0.384 0.616]
q(y) = [0.16 0.84]
q(y') = [0.16 0.84]
q(y") = [0.16 0.84]
q(z) = [0.4 0.6]
